split_native_kaggle.py
Run SPLIT (Hyun & Kim, ECCV 2026) with its OFFICIAL code and DEFAULT settings
on FakeParts and ViF-Bench, subset by subset, on Kaggle.

Uses the official repository  github.com/mldljyh/SPLIT  (pinned commit below)
for the model and the frame reader, so the score is exactly SPLIT's.
Only the frame-extraction step is re-implemented, because the official script
uses a 32-process pool and moviepy; the settings are identical
(8 fps, 3-second window, random start for videos longer than 3 s, JPEG frames).

Each "# %%" block is one Kaggle cell. Turn on a GPU (T4 or P100) and Internet.

  Cell 1  settings            -> edit DATASET, SUBSET_NAME, SUBSET_DIR, LABEL
  Cell 2  setup + functions   -> once per session
  Cell 3  score this subset   -> saves /kaggle/working/split_native/<DATASET>/<SUBSET>.csv
  Cell 4  (optional) delete the subset's videos
  ...repeat cells 1, 3, 4 for every subset...
  Cell 5  final metrics       -> once, after all subsets of a benchmark

/kaggle/working is erased when the session ends: "Save Version" after each
subset, or download the per-subset CSVs (they are small).

In [ ]:
# %% --------------------------------------------------------- CELL 1: SETTINGS
DATASET     = "FakeParts"                 # "FakeParts" or "ViF-Bench"
SUBSET_NAME = "inpainting"                # short name; used as category in the results
SUBSET_DIR  = "/kaggle/working/data/inpainting"   # folder with this subset's videos
LABEL       = 1                           # 0 = real, 1 = generated / edited

ENCODER     = "XCLIP-16"                  # SPLIT default; do not change for the main run
OUT_ROOT    = "/kaggle/working/split_native"
FRAMES_TMP  = "/kaggle/working/split_frames"   # temporary JPEG frames, deleted per video
SEED        = 42
START_MODE  = "random"                    # "random" = SPLIT default (random 3-s window,
                                          #   seeded per video so it is reproducible)
                                          # "start"  = always the first 3 s (sensitivity check)
EXTRACT_WORKERS = 4                       # parallel ffmpeg processes

# Video id used to merge with your own per-video scores later.
# Default: path relative to SUBSET_DIR without extension, prefixed by the subset.
def VIDEO_ID(rel_path_without_ext):
    return f"{SUBSET_NAME}/{rel_path_without_ext}"

In [ ]:
# %% ------------------------------------------------- CELL 2: SETUP + FUNCTIONS
import os, sys, subprocess, shutil, hashlib, random, json, time, csv
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

SPLIT_REPO   = "/kaggle/working/SPLIT"
SPLIT_COMMIT = "062820b4db407e69e94ef23ae5feb59a5a621fd0"   # official code, 2 Nov 2026

if not Path(SPLIT_REPO).exists():
    subprocess.run(["git", "clone", "https://github.com/mldljyh/SPLIT", SPLIT_REPO], check=True)
subprocess.run(["git", "-C", SPLIT_REPO, "checkout", "-q", SPLIT_COMMIT], check=True)
# Kaggle already has torch / transformers / opencv. Only albumentations may be missing.
try:
    import albumentations  # noqa
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "albumentations"], check=True)
if shutil.which("ffmpeg") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, check=True)

sys.path.insert(0, SPLIT_REPO)
import numpy as np
import torch
from models import SPLIT_model                                   # official model
from data.datasets import read_video, set_preprocessing          # official reader:
#   first 16 frames (8 if fewer than 16), 10% crop on each side of the longer
#   dimension, resize 224x224, ImageNet normalisation

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".gif", ".webm", ".m4v"}
FIELDS = ["video_id", "subset", "label", "relpath", "duration_s", "start_s",
          "frames_extracted", "frames_used", "score", "status"]

_model = None
def get_model():
    global _model
    if _model is None:
        torch.manual_seed(SEED)
        _model = SPLIT_model(encoder_type=ENCODER, loss_type="l2").cuda().eval()
    return _model


def video_duration(path):
    """Duration in seconds via ffprobe (the official script uses moviepy)."""
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                        "-of", "default=nw=1:nk=1", str(path)], capture_output=True, text=True)
    try:
        return float(r.stdout.strip())
    except ValueError:
        return float("nan")


def extract_frames(path, out_dir, vid):
    """SPLIT's utils/video2frame.py: 8 fps over a 3-s window starting at a
    random integer second when the video is longer than 3 s, else at 0."""
    dur = video_duration(path)
    if START_MODE == "random" and dur == dur and dur > 3:
        rng = random.Random(int(hashlib.md5(f"{SEED}:{vid}".encode()).hexdigest(), 16))
        start = int(np.floor(rng.uniform(0, dur - 3)))
    else:
        start = 0
    out_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["ffmpeg", "-loglevel", "quiet", "-ss", str(start), "-t", "3", "-i", str(path),
                    "-vf", "fps=8", str(out_dir / "%d.jpg")], check=False)
    n = len([f for f in os.listdir(out_dir) if f.endswith(".jpg")])
    return dur, start, n


def score_subset():
    root = Path(SUBSET_DIR)
    files = sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS)
    if not files:
        raise RuntimeError(f"No videos under {root}")
    out_csv = Path(OUT_ROOT) / DATASET / f"{SUBSET_NAME}.csv"
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    done = {}
    if out_csv.exists():                       # resume after an interruption
        with open(out_csv) as f:
            done = {r["video_id"]: r for r in csv.DictReader(f)}
    model, trans = get_model(), set_preprocessing(None, None)
    todo = []
    for p in files:
        rel = p.relative_to(root)
        vid = VIDEO_ID(str(rel.with_suffix("")))
        if vid not in done:
            todo.append((p, rel, vid))
    print(f"[{DATASET}/{SUBSET_NAME}] {len(files)} videos, {len(done)} already scored, {len(todo)} to do")

    fh = open(out_csv, "a", newline="")
    w = csv.DictWriter(fh, fieldnames=FIELDS)
    if not done:
        w.writeheader()

    def prep(item):
        p, rel, vid = item
        d = Path(FRAMES_TMP) / hashlib.md5(vid.encode()).hexdigest()
        if d.exists():
            shutil.rmtree(d)
        dur, start, n = extract_frames(p, d, vid)
        return p, rel, vid, d, dur, start, n

    t0 = time.time()
    with ThreadPoolExecutor(max_workers=EXTRACT_WORKERS) as ex:
        for k, (p, rel, vid, d, dur, start, n) in enumerate(ex.map(prep, todo), 1):
            row = dict(video_id=vid, subset=SUBSET_NAME, label=LABEL, relpath=str(rel),
                       duration_s=round(dur, 3) if dur == dur else "", start_s=start,
                       frames_extracted=n, frames_used="", score="", status="")
            try:
                if n < 8:                                   # SPLIT drops these videos
                    row["status"] = "excluded_fewer_than_8_frames"
                else:
                    x = read_video(str(d), trans=trans)     # (T, 3, 224, 224), T = 16 or 8
                    with torch.no_grad():
                        s = model.forward_score(x.unsqueeze(0).cuda())
                    row.update(frames_used=x.shape[0], score=float(s.flatten()[0]), status="ok")
            except Exception as e:
                row["status"] = f"error: {str(e)[:120]}"
            w.writerow(row)
            shutil.rmtree(d, ignore_errors=True)
            if k % 200 == 0 or k == len(todo):
                fh.flush()
                print(f"  {k}/{len(todo)}  ({(time.time() - t0) / k:.2f} s/video)", flush=True)
    fh.close()
    print("Saved", out_csv)


# ------------------------------------------------------------------ metrics
def _auroc(real, fake):
    from sklearn.metrics import roc_auc_score
    y = np.r_[np.zeros(len(real)), np.ones(len(fake))]
    return roc_auc_score(y, np.r_[real, fake])


def _recall_at_fpr(real, fake, fpr):
    thr = np.quantile(real, 1 - fpr, method="higher")      # same rule as SPLIT eval.py
    return float((fake >= thr).mean())


def _metrics(real, fake):
    from sklearn.metrics import average_precision_score
    y = np.r_[np.zeros(len(real)), np.ones(len(fake))]
    s = np.r_[real, fake]
    return {"n_real": len(real), "n_fake": len(fake),
            "AUROC": 100 * _auroc(real, fake),
            "AP_fake": 100 * average_precision_score(y, s),
            "recall@0.1%FPR": 100 * _recall_at_fpr(real, fake, 0.001),
            "recall@1%FPR": 100 * _recall_at_fpr(real, fake, 0.01),
            "recall@5%FPR": 100 * _recall_at_fpr(real, fake, 0.05)}


def final_metrics(primary_csv=None, primary_id_col="video_id", primary_score_col="score"):
    """Pooled and per-subset metrics; optionally complementarity with your
    primary cross-encoder score (a CSV with one row per video)."""
    import pandas as pd
    d = Path(OUT_ROOT) / DATASET
    df = pd.concat([pd.read_csv(p) for p in sorted(d.glob("*.csv")) if p.name != "SUMMARY.csv"],
                   ignore_index=True)
    print("Status counts:\n", df.groupby(["subset", "status"]).size().to_string(), "\n")
    ok = df[df.status == "ok"].copy()
    real = ok[ok.label == 0].score.to_numpy()
    rows = [{"group": "ALL", **_metrics(real, ok[ok.label == 1].score.to_numpy())}]
    for sub, g in ok[ok.label == 1].groupby("subset"):
        rows.append({"group": sub, **_metrics(real, g.score.to_numpy())})
    res = pd.DataFrame(rows).round(2)
    print(res.to_string(index=False))

    if primary_csv:
        from scipy.stats import spearmanr
        pr = pd.read_csv(primary_csv)[[primary_id_col, primary_score_col]]
        pr.columns = ["video_id", "primary"]
        m = ok.merge(pr, on="video_id", how="inner")
        print(f"\nMerged with primary score: {len(m)} of {len(ok)} videos")
        r_ref = m[m.label == 0]
        def pct(v, ref):                       # percentile of the real-video distribution
            ref = np.sort(ref)
            return np.searchsorted(ref, v, side="right") / len(ref)
        m["comb"] = (pct(m.score, r_ref.score) + pct(m.primary, r_ref.primary)) / 2
        rr, ff = m[m.label == 0], m[m.label == 1]
        comp = {"spearman_rho": spearmanr(m.score, m.primary)[0],
                "AUROC_SPLIT": 100 * _auroc(rr.score, ff.score),
                "AUROC_primary": 100 * _auroc(rr.primary, ff.primary),
                "AUROC_combined": 100 * _auroc(rr.comb, ff.comb)}
        print({k: round(v, 3) for k, v in comp.items()})
        res = pd.concat([res, pd.DataFrame([{"group": "complementarity", **comp}])])
    res.to_csv(d / "SUMMARY.csv", index=False)
    print("\nSaved", d / "SUMMARY.csv")
    return res


print("SPLIT", SPLIT_COMMIT[:8], "loaded | encoder", ENCODER)

In [ ]:
# %% ------------------------------------------------- CELL 3: SCORE THIS SUBSET
score_subset()

In [ ]:
# %% ---------------------------------------- CELL 4 (optional): DELETE SUBSET
# shutil.rmtree(SUBSET_DIR); print("Deleted", SUBSET_DIR)

In [ ]:
# %% --------------------------------- CELL 5: FINAL METRICS (after all subsets)
# final_metrics()
# With your primary scores (one row per video, same video ids):
# final_metrics(primary_csv="/kaggle/input/<your-data>/primary_scores.csv",
#               primary_id_col="video_id", primary_score_col="cka")